# AI, ML & Database RAG Assistant

This notebook builds and evaluates a grounded RAG pipeline from a text-document collection. Run it top-to-bottom after running `python scripts/download_data.py` from the project root.

## 2.1 Load and Inspect Documents

The corpus contains three UTF-8 text files: AI, Machine Learning, and Database notes. This version uses `.txt` files, so no PDF parsing or OCR is required. The inspection cell reports files that fail to decode.

In [1]:
from pathlib import Path
import json
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "data" / "raw").exists()), cwd.parent)
RAW_DIR = PROJECT_ROOT / "data" / "raw"
VECTOR_DIR = PROJECT_ROOT / "backend" / "data" / "vector_store"
RAW_DIR, VECTOR_DIR

(WindowsPath('C:/Users/youss/Desktop/ITI_RAG/rag_ai_ml_database_assistant/data/raw'),
 WindowsPath('C:/Users/youss/Desktop/ITI_RAG/rag_ai_ml_database_assistant/backend/data/vector_store'))

In [2]:
documents = []
failures = []
for path in sorted(RAW_DIR.glob("*.txt")):
    try:
        text = path.read_text(encoding="utf-8").strip()
        documents.append({"source": path.name, "text": text, "characters": len(text)})
    except Exception as exc:
        failures.append({"source": path.name, "error": str(exc)})

inspection = pd.DataFrame(documents)[["source", "characters"]]
print(f"Documents loaded: {len(documents)}")
print(f"Failed files: {len(failures)}")
display(inspection)
if failures:
    display(pd.DataFrame(failures))
assert documents, "No documents found. Run python scripts/download_data.py from the project root."

Documents loaded: 3
Failed files: 0


,source,characters
0,AI.txt,3642
1,Database.txt,7958
2,Machine_Learning.txt,5625


## 2.2 Chunking Strategy

The system uses fixed-size chunks of 800 characters with 150 characters of overlap. This chunk size retains enough local explanation for introductory technical concepts while remaining focused for semantic retrieval. Overlap prevents important sentences at chunk boundaries from losing context.

In [3]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    text = " ".join(text.split())
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start + chunk_size // 2:
                end = boundary
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

chunks = []
for doc in documents:
    for chunk_id, text in enumerate(chunk_text(doc["text"])):
        chunks.append({"id": f"{doc['source']}_{chunk_id}", "text": text, "metadata": {"source": doc["source"], "chunk_id": chunk_id}})
print(f"Created {len(chunks)} chunks")
pd.DataFrame([{**c["metadata"], "characters": len(c["text"])} for c in chunks]).head()

Created 28 chunks


,source,chunk_id,characters
0,AI.txt,0,796
1,AI.txt,1,792
2,AI.txt,2,793
3,AI.txt,3,796
4,AI.txt,4,799


## 2.3 Embeddings and Vector Store

Each chunk is embedded with `sentence-transformers/all-MiniLM-L6-v2`. ChromaDB persists both embeddings and metadata under `backend/data/vector_store`, so the FastAPI service can load it directly without rebuilding the index.

In [4]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "ai_ml_database_documents"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

model = SentenceTransformer(EMBEDDING_MODEL)
embeddings = model.encode([chunk["text"] for chunk in chunks], normalize_embeddings=True, show_progress_bar=True).tolist()

client = chromadb.PersistentClient(path=str(VECTOR_DIR))
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(COLLECTION_NAME, metadata={"description": "AI, ML and Database educational text corpus"})
collection.add(
    ids=[chunk["id"] for chunk in chunks],
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[chunk["metadata"] for chunk in chunks],
    embeddings=embeddings,
)
config = {"embedding_model": EMBEDDING_MODEL, "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP, "collection_name": COLLECTION_NAME, "document_count": len(documents), "chunk_count": len(chunks)}
(VECTOR_DIR / "rag_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"Persisted {collection.count()} chunks to {VECTOR_DIR}")
config

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\youss\Desktop\ITI_RAG\rag_ai_ml_database_assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\youss\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Persisted 28 chunks to C:\Users\youss\Desktop\ITI_RAG\rag_ai_ml_database_assistant\backend\data\vector_store


{'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'chunk_size': 800,
 'chunk_overlap': 150,
 'collection_name': 'ai_ml_database_documents',
 'document_count': 3,
 'chunk_count': 28}

## 2.4 Retrieval and Prompting

Retrieved context is supplied to Ollama with an instruction to answer only from that context. If no support exists, the assistant must refuse rather than use external knowledge. Source metadata is returned with each answer.

In [5]:
def retrieve(question: str, top_k: int = 4) -> list[dict]:
    query_embedding = model.encode(question, normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=[query_embedding], n_results=min(top_k, collection.count()), include=["documents", "metadatas", "distances"])
    return [{"text": d, "metadata": m, "distance": dist} for d, m, dist in zip(result["documents"][0], result["metadatas"][0], result["distances"][0])]

def build_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    context = "\n\n".join(f"[Source: {c['metadata']['source']} | Chunk: {c['metadata']['chunk_id']}]\n{c['text']}" for c in retrieved_chunks)
    return f"""You are a grounded academic assistant. Answer only using the Context below. If the answer is absent, say exactly: I could not find this information in the provided documents. Do not use outside knowledge.

Context:
{context}

Question: {question}

Answer:"""

question = "What is machine learning?"
retrieved = retrieve(question)
for item in retrieved:
    print(f"{item['metadata']['source']} — chunk {item['metadata']['chunk_id']} — distance {item['distance']:.3f}")
    print(item['text'][:250], "\n")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Machine_Learning.txt — chunk 0 — distance 0.500
Machine Learning - Complete Guide Introduction to Machine Learning Machine Learning (ML) is a subset of artificial intelligence that focuses on building systems that can learn from and make decisions based on data. Instead of being explicitly program 

Machine_Learning.txt — chunk 1 — distance 0.814
from data Types of Machine Learning Supervised Learning In supervised learning, the algorithm learns from labeled training data to make predictions or decisions. - Regression: Predicting continuous values (e.g., house prices, temperature) - Classific 

AI.txt — chunk 2 — distance 0.909
vity, general wisdom, and problem-solving. This is a theoretical concept. Key Components of AI - Machine Learning: The ability of systems to learn and improve from experience without being explicitly programmed. - Neural Networks: Computing systems i 

Machine_Learning.txt — chunk 3 — distance 1.026
s The Machine Learning Workflow 1. Problem Definition: Clearly 

In [6]:
# Optional live Ollama test. Run only after `ollama pull llama3.2:3b`.
# import ollama
# response = ollama.chat(model="llama3.2:3b", messages=[{"role": "user", "content": build_prompt(question, retrieved)}], options={"temperature": 0})
# print(response["message"]["content"])

## 2.5 Vision Component

This project follows the Core Track. It is a text-only RAG assistant, so a YOLO or computer-vision component is not included.

## 2.6 Evaluation

Evaluate retrieval relevance and grounding for at least ten questions. Run each question through the FastAPI/Streamlit application after the vector store is created, then fill the result fields honestly using the returned sources and answer.

In [7]:
evaluation_questions = [
    "What is artificial intelligence?",
    "What is machine learning?",
    "What is the difference between supervised and unsupervised learning?",
    "What is overfitting in machine learning?",
    "What is a database?",
    "What is a DBMS?",
    "What are the advantages of using a database?",
    "How are AI and machine learning related?",
    "What is the purpose of training data?",
    "What is the capital of Egypt?",
]

rows = []
for question in evaluation_questions:
    hits = retrieve(question)
    rows.append({
        "question": question,
        "retrieved_source": "; ".join(f"{hit['metadata']['source']} (chunk {hit['metadata']['chunk_id']})" for hit in hits),
        "answer_summary": "Run with Ollama and record actual answer",
        "context_relevant": "To be evaluated",
        "grounded_or_hallucinated": "To be evaluated",
        "correct_or_refusal": "To be evaluated",
    })
evaluation_df = pd.DataFrame(rows)
display(evaluation_df)
evaluation_df.to_csv(PROJECT_ROOT / "evaluation_template.csv", index=False)

,question,retrieved_source,answer_summary,context_relevant,grounded_or_hallucinated,correct_or_refusal
0,What is artificial intelligence?,AI.txt (chunk 0); AI.txt (chunk 1); AI.txt (ch...,Run with Ollama and record actual answer,To be evaluated,To be evaluated,To be evaluated
1,What is machine learning?,Machine_Learning.txt (chunk 0); Machine_Learni...,Run with Ollama and record actual answer,To be evaluated,To be evaluated,To be evaluated
2,What is the difference between supervised and ...,Machine_Learning.txt (chunk 1); Machine_Learni...,Run with Ollama and record actual answer,To be evaluated,To be evaluated,To be evaluated
3,What is overfitting in machine learning?,Machine_Learning.txt (chunk 5); Machine_Learni...,Run with Ollama and record actual answer,To be evaluated,To be evaluated,To be evaluated
4,What is a database?,Database.txt (chunk 0); Database.txt (chunk 1)...,Run with Ollama and record actual answer,To be evaluated,To be evaluated,To be evaluated
5,What is a DBMS?,Database.txt (chunk 0); Database.txt (chunk 1)...,Run with Ollama and record actual answer,To be evaluated,To be evaluated,To be evaluated
6,What are the advantages of using a database?,Database.txt (chunk 0); Database.txt (chunk 1)...,Run with Ollama and record actual answer,To be evaluated,To be evaluated,To be evaluated
7,How are AI and machine learning related?,AI.txt (chunk 0); Machine_Learning.txt (chunk ...,Run with Ollama and record actual answer,To be evaluated,To be evaluated,To be evaluated
8,What is the purpose of training data?,Machine_Learning.txt (chunk 0); Machine_Learni...,Run with Ollama and record actual answer,To be evaluated,To be evaluated,To be evaluated
9,What is the capital of Egypt?,Database.txt (chunk 8); Database.txt (chunk 11...,Run with Ollama and record actual answer,To be evaluated,To be evaluated,To be evaluated


### Failure cases and mitigation

Expected failure cases include broad questions that require information across multiple chunks, concepts absent from the small corpus, and semantically similar AI/ML terminology. Mitigations are chunk overlap, top-k retrieval, source metadata, and a strict prompt that requires refusal when the context does not support an answer. After live testing, replace this paragraph with observations from your actual 10-question evaluation.

## 2.7 Export

The previous embedding cell exported the persisted ChromaDB collection and a `rag_config.json` file to `backend/data/vector_store`. The FastAPI backend loads this folder directly; it never rebuilds the vector store at request time.